In [0]:
important_posts = spark.sql("""
SELECT id AS post_id, customer_id AS author_user_id
FROM datalake_community.dw_cust_community_tb_post
WHERE is_hide = 0
  AND audit_status = 1
  AND publish_approval_status IN (2,3)
  AND delete_at IS NULL
  AND post_attribute = 1
  AND is_sink = 0
  AND is_top_forum = 1
ORDER BY from_unixtime(post_time / 1000) DESC
LIMIT 5
""").collect()

IMPORTANT_POST_IDS = [r["post_id"] for r in important_posts]
IMPORTANT_POST_AUTHORS = {r["post_id"]: r["author_user_id"] for r in important_posts}

print("IMPORTANT_POST_IDS:", IMPORTANT_POST_IDS)

In [0]:
%%sql
-- 1) Minimal lookup for author by post_id (add any other fields you need)
CREATE or REPLACE TABLE revr_bmbs_dmp_offline.post_author_lookup
USING delta
AS
SELECT
  id AS post_id,
  customer_id AS author_user_id
FROM bz_cn_dl.datalake_community.dw_cust_community_tb_post
WHERE is_hide = 0
  AND delete_at IS NULL;

In [0]:
# ============================================================
# PURPOSE
#   Build per-user JSON payloads from content ranking results with rules:
#     Rule0: Output must be exactly 50 posts (keep top-ranked)
#     Rule1: Avoid 3 consecutive posts from the same author (best-effort swap)
#     Rule2: Ensure  "important posts" is in the first  positionss:
#            - If already present in top 60: MOVE it into top 3 (no duplicates)
#            - If not present: INSERT into top 3 (then trim back to 60)
#
#   Then choose one mode:
#     - MODE="PRINT": show sample output (no SQS send)
#     - MODE="SEND" : send to AWS SQS FIFO using outbox + send-log checkpointing
#
# OPERATIONAL NOTES
#   - Outbox prevents duplicates on normal reruns by sending only PENDING rows.
#   - If you cancel mid-run, some messages may reach SQS but not yet be logged.
#     Rerun may re-send a subset (at-least-once behavior).
# ============================================================

import hashlib
import uuid
import json
import pandas as pd
from pyspark.sql import functions as F, types as T

# ------------------------------------------------------------
# CONFIG (change only this section day-to-day)
# ------------------------------------------------------------

# Create widgets with defaults
dbutils.widgets.text("MODE", "PRINT")
dbutils.widgets.text("PROFILE", "DEV")
dbutils.widgets.text("run_date", "")

# Read values passed from ADF
MODE = dbutils.widgets.get("MODE").strip().upper()
PROFILE = dbutils.widgets.get("PROFILE").strip().upper()
RUN_DATE = dbutils.widgets.get("run_date").strip()

#MODE = "SEND"                 # "PRINT" or "SEND"
PRINT_N = 10                   # rows to display in PRINT mode
SHOW_FULL_JSON = True          # True => truncate=False, False => truncate=200
SAMPLE_USERS = None            # e.g. ["u01","u02"] or None

#PROFILE = "PROD"                # "Dev" or "UAT" or "PROD"
RAND_SEED = 42                 # deterministic position per user (1..3)

# Tables (switch to prod when ready)
SOURCE_TABLE = "revr_bmbs_dmp_offline.content_recom_ranking_results" # change to actual table 
AUTHOR_LOOKUP_TABLE = "revr_bmbs_dmp_offline.post_author_lookup"  # must contain: post_id, author_user_id

# SQS / Outbox tables (only for MODE="SEND")
OUTBOX_TABLE  = "revr_bmbs_dmp_offline.sqs_outbox_content_reco"
SENDLOG_TABLE = "revr_bmbs_dmp_offline.sqs_outbox_send_log"

# Tunables
NUM_GROUPS  = 1024
BATCH_SIZE  = 10
MAX_RETRIES = 5
TARGET_PARTITIONS = 256
SQS_MAX_BYTES = 256 * 1024
TARGET_POSTS = 60  # FINAL must be exactly 60

# ------------------------------------------------------------
# Spark configs
# ------------------------------------------------------------
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# ------------------------------------------------------------
# ENVIRONMENT CONFIGURATION
# ------------------------------------------------------------
ENV_CONFIG = {
    "DEV": {
        "queue_url": "https://sqs.cn-northwest-1.amazonaws.com.cn/634001137982/dmp-mq-dev.fifo",
        "aws_region": "cn-northwest-1",
        "aws_access_key_id_secret": "XDLK-ONEAPP-AWS-ACCESS-KEY-ID",
        "aws_secret_access_key_secret": "XDLK-ONEAPP-AWS-SECRET-ACCESS-KEY",
    },
    "UAT": {
        "queue_url": "https://sqs.cn-northwest-1.amazonaws.com.cn/634001137982/dmp-mq-uat.fifo",
        "aws_region": "cn-northwest-1",
        "aws_access_key_id_secret": "XDLK-ONEAPP-AWS-ACCESS-KEY-ID",
        "aws_secret_access_key_secret": "XDLK-ONEAPP-AWS-SECRET-ACCESS-KEY",
    },
    "PROD": {
        "queue_url": "https://sqs.cn-northwest-1.amazonaws.com.cn/633869915042/dmp-mq.fifo",
        "aws_region": "cn-northwest-1",
        "aws_access_key_id_secret": "PDLK-ONEAPP-AWS-ACCESS-KEY-ID",
        "aws_secret_access_key_secret": "PDLK-ONEAPP-AWS-SECRET-ACCESS-KEY",
    },
}

# ------------------------------------------------------------
# LOAD ENVIRONMENT SETTINGS
# ------------------------------------------------------------
PROFILE_CLEAN = (PROFILE or "").strip().upper()
MODE_CLEAN = (MODE or "").strip().upper()

if PROFILE_CLEAN not in ENV_CONFIG:
    raise ValueError(f"Unknown PROFILE: {PROFILE}. Allowed values: {list(ENV_CONFIG.keys())}")

env = ENV_CONFIG[PROFILE_CLEAN]

QUEUE_URL = env["queue_url"]
AWS_REGION = env["aws_region"]

AWS_ACCESS_KEY_ID = dbutils.secrets.get(
    scope="dmp_secrets",
    key=env["aws_access_key_id_secret"]
)

AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(
    scope="dmp_secrets",
    key=env["aws_secret_access_key_secret"]
)

RUN_ID = str(uuid.uuid4())

print(f"[INFO] MODE={MODE_CLEAN} | PROFILE={PROFILE_CLEAN} | RUN_ID={RUN_ID}")
print(f"[INFO] QUEUE_URL={QUEUE_URL}")
print(f"[INFO] AWS_REGION={AWS_REGION}")
print(f"[INFO] RUN_DATE={RUN_DATE}")
# ============================================================
# STEP 0: IMPORTANT POST (REAL)
# ============================================================
IMPORTANT_N = 5  # max 5 now; can change later. // change to PROD TABLE

important_rows = spark.sql(f"""
SELECT id AS post_id, customer_id AS author_user_id
FROM 
--bz_cn_dl.revr_bmbs_dmp_offline.post_table_uat
datalake_community.dw_cust_community_tb_post
WHERE is_hide = 0
  AND audit_status = 1
  AND publish_approval_status IN (2,3)
  AND delete_at IS NULL
  AND post_attribute = 1
  AND is_sink = 0
  AND is_top_forum = 1
ORDER BY from_unixtime(post_time / 1000) DESC
LIMIT {IMPORTANT_N}
""").collect()

IMPORTANT_POST_IDS = [r["post_id"] for r in important_rows]   # keep SQL order
IMPORTANT_AUTHORS  = {r["post_id"]: r["author_user_id"] for r in important_rows}

print(f"[INFO] IMPORTANT_POST_IDS={IMPORTANT_POST_IDS}")

# ============================================================
# STEP 1: Requested partition_date
# ============================================================
if not RUN_DATE:
    raise ValueError("run_date is required")

partition_exists = spark.sql(f"""
SELECT 1 AS found
FROM {SOURCE_TABLE}
WHERE partition_date = '{RUN_DATE}'
LIMIT 1
""").first()

if not partition_exists:
    raise RuntimeError(f"No data found in {SOURCE_TABLE} for partition_date={RUN_DATE}")

TARGET_PARTITION = RUN_DATE

print(f"[INFO] TARGET_PARTITION={TARGET_PARTITION}")

# ============================================================
# HELPERS
# ============================================================
def _rand_pos_1_to_3(meid: str, seed: int) -> int:
    """Deterministic 'random' position in [1..3] per user."""
    h = hashlib.md5(f"{meid}-{seed}".encode("utf-8")).hexdigest()
    return (int(h, 16) % 3) + 1

def _fix_no_three(items):
    """
    Best-effort:
      If 3 consecutive same author exists, swap the 3rd with the next different-author post.
      If no alternative exists, the triple remains (cannot be fixed).
    """
    i = 0
    while i < len(items):
        if i >= 2:
            a1 = items[i]["author_user_id"]
            a2 = items[i-1]["author_user_id"]
            a3 = items[i-2]["author_user_id"]
            if a1 == a2 == a3:
                j = i + 1
                while j < len(items) and items[j]["author_user_id"] == a1:
                    j += 1
                if j < len(items):
                    items[i], items[j] = items[j], items[i]
                    i = max(i - 2, 0)
                    continue
        i += 1
    return items

def _group_for(meid: str) -> str:
    shard = int(hashlib.md5(meid.encode("utf-8")).hexdigest(), 16) % NUM_GROUPS
    return f"dmp-group-{shard}"

def _dedup_for(message_hash: str) -> str:
    return message_hash

# ============================================================
# STEP 2: Build base_input = one row per meid with items array
# ============================================================
user_filter_sql = ""
if SAMPLE_USERS:
    in_list = ",".join([f"'{u}'" for u in SAMPLE_USERS])
    user_filter_sql = f" AND user_id IN ({in_list}) "

src = spark.sql(f"""
SELECT user_id AS meid, post_id
FROM {SOURCE_TABLE}
WHERE partition_date = '{TARGET_PARTITION}'
  AND user_id IS NOT NULL
  AND post_id IS NOT NULL
  AND size(post_id) > 0
  {user_filter_sql}
""")

exploded = (
    src.select("meid", F.posexplode("post_id").alias("pos", "postid"))
       .withColumn("seq", F.col("pos") + F.lit(1))
)

author_lkp = spark.table(AUTHOR_LOOKUP_TABLE)

joined = (
    exploded.join(author_lkp, exploded.postid == author_lkp.post_id, "left")
            .select(
                "meid",
                "seq",
                F.col("postid").alias("post_id"),
                F.coalesce(F.col("author_user_id"), F.lit("UNKNOWN")).alias("author_user_id")
            )
)

base_input = (
    joined.groupBy("meid")
          .agg(F.sort_array(
               F.collect_list(F.struct("seq","post_id","author_user_id"))
          ).alias("items"))
          .withColumn("item_count", F.size("items"))
          .filter(F.col("item_count") >= TARGET_POSTS)
)

# ============================================================
# STEP 3: applyInPandas rules -> (partition_date, meid, message_body)
# ============================================================
out_schema = T.StructType([
    T.StructField("partition_date", T.StringType(), False),
    T.StructField("meid", T.StringType(), False),
    T.StructField("message_body", T.StringType(), False),
])

def apply_rules(pdf: pd.DataFrame) -> pd.DataFrame:
    row = pdf.iloc[0]
    meid = row["meid"]
    items = row["items"]

    items = [{
        "seq": int(x["seq"]),
        "post_id": x["post_id"],
        "author_user_id": x["author_user_id"]
    } for x in items]

    # Rule0: normalize to top 50
    if len(items) > TARGET_POSTS:
        items = items[:TARGET_POSTS]

    # Rule1
    items = _fix_no_three(items)

    # Rule2: Promote ALL important posts to the top (no duplicates)
    #   - Keeps the same order as IMPORTANT_POST_IDS (from SQL ORDER BY ... DESC)
    important_set = set(IMPORTANT_POST_IDS)

    # Remove any existing occurrences of important posts (avoid duplicates)
    tail = [x for x in items if x["post_id"] not in important_set]

    # Build the "top" block in the exact SQL order
    top = []
    for pid in IMPORTANT_POST_IDS:
        top.append({
            "seq": 0,
            "post_id": pid,
            "author_user_id": str(IMPORTANT_AUTHORS.get(pid, "UNKNOWN"))
        })

    # Put all important posts first, then the remaining ranked posts
    items = top + tail

    # Enforce exactly 50 after move/insert
    if len(items) > TARGET_POSTS:
        items = items[:TARGET_POSTS]

    # Rule1 again (post-adjustment)
    items = _fix_no_three(items)

    if len(items) != TARGET_POSTS:
        raise ValueError(f"Final post count != {TARGET_POSTS} for meid={meid}. Got={len(items)}")

    postIDs = [{"postID": x["post_id"], "sequence": i + 1} for i, x in enumerate(items)]
    message_body = json.dumps({"meID": meid, "postIDs": postIDs}, ensure_ascii=False)

    return pd.DataFrame([{
        "partition_date": TARGET_PARTITION,
        "meid": meid,
        "message_body": message_body
    }])

base = base_input.groupBy("meid").applyInPandas(apply_rules, schema=out_schema)

base = (
    base
    .withColumn("message_hash", F.sha2(F.col("message_body"), 256))
    .withColumn("message_size", F.length("message_body"))
)

# ============================================================
# MODE SWITCH
# ============================================================
if MODE_CLEAN == "PRINT":
    print(f"[INFO] PRINT mode: showing {PRINT_N} rows (partition={TARGET_PARTITION})")

    display_df = (
        base.select("partition_date","meid","message_size","message_hash","message_body")
            .orderBy("meid")
            .limit(PRINT_N)
    )

    if SHOW_FULL_JSON:
        display_df.show(PRINT_N, truncate=False)
    else:
        display_df.show(PRINT_N, truncate=200)

    print("[INFO] PRINT mode completed. No SQS messages sent.")

elif MODE_CLEAN == "SEND":
    # ---- create tables if needed ----
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {OUTBOX_TABLE} (
      partition_date STRING,
      meid           STRING,
      message_body   STRING,
      message_hash   STRING,
      status         STRING,
      run_id         STRING,
      created_ts     TIMESTAMP,
      sent_ts        TIMESTAMP
    )
    USING delta
    PARTITIONED BY (partition_date)
    """)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SENDLOG_TABLE} (
      partition_date STRING,
      meid           STRING,
      message_hash   STRING,
      run_id         STRING,
      sent_ts        TIMESTAMP
    )
    USING delta
    PARTITIONED BY (partition_date)
    """)

    # ---- outbox rows ----
    outbox_rows = (
        base.where(F.col("message_size") <= SQS_MAX_BYTES)
            .select(
                "partition_date","meid","message_body","message_hash",
                F.lit("PENDING").alias("status"),
                F.lit(RUN_ID).alias("run_id"),
                F.current_timestamp().alias("created_ts")
            )
    )
    outbox_rows.createOrReplaceTempView("new_outbox_rows")

    spark.sql(f"""
    MERGE INTO {OUTBOX_TABLE} t
    USING new_outbox_rows s
    ON  t.partition_date = s.partition_date
    AND t.meid           = s.meid
    WHEN NOT MATCHED THEN
      INSERT (partition_date, meid, message_body, message_hash, status, run_id, created_ts, sent_ts)
      VALUES (s.partition_date, s.meid, s.message_body, s.message_hash, s.status, s.run_id, s.created_ts, CAST(NULL AS TIMESTAMP))
    """)

    pending_df = spark.sql(f"""
    SELECT partition_date, meid, message_body, message_hash
    FROM {OUTBOX_TABLE}
        WHERE partition_date = '{TARGET_PARTITION}'
      AND status = 'PENDING'
    """)

    df = (
        pending_df
        .withColumn("message_size", F.length("message_body"))
        .where(F.col("message_size") <= SQS_MAX_BYTES)
        .repartition(TARGET_PARTITIONS, F.col("meid"))
        .select("partition_date","meid","message_body","message_hash")
    )

    total_to_send = df.count()
    print(f"[INFO] SEND mode: pending messages to send = {total_to_send:,}")
    if total_to_send == 0:
        raise SystemExit("[INFO] Nothing to send. Exiting.")

    sendlog_schema = T.StructType([
        T.StructField("partition_date", T.StringType(), False),
        T.StructField("meid", T.StringType(), False),
        T.StructField("message_hash", T.StringType(), False),
        T.StructField("run_id", T.StringType(), False),
        T.StructField("sent_ts", T.TimestampType(), False),
    ])

    def send_map_in_pandas(pdf_iter):
        import boto3, time
        from datetime import datetime

        sqs = boto3.client(
            "sqs",
            region_name=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
            endpoint_url=f"https://sqs.{AWS_REGION}.amazonaws.com.cn"
        )

        def flush_batch(batch, batch_meta):
            if not batch:
                return []
            backoff = 0.25
            to_send, to_send_meta = batch, batch_meta

            for _ in range(MAX_RETRIES):
                resp = sqs.send_message_batch(QueueUrl=QUEUE_URL, Entries=to_send)
                failed = resp.get("Failed", [])
                if not failed:
                    return to_send_meta

                retry_ids = {f["Id"] for f in failed}
                new_to_send, new_meta = [], []
                for e, m in zip(to_send, to_send_meta):
                    if e["Id"] in retry_ids:
                        new_to_send.append(e)
                        new_meta.append(m)
                to_send, to_send_meta = new_to_send, new_meta

                time.sleep(backoff)
                backoff *= 2

            raise RuntimeError("send_message_batch failed after retries")

        for pdf in pdf_iter:
            batch, meta, local_id = [], [], 0
            success_rows = []

            for _, row in pdf.iterrows():
                local_id += 1
                partition_date = row["partition_date"]
                meid = row["meid"]
                body = row["message_body"]
                message_hash = row["message_hash"]

                batch.append({
                    "Id": str(local_id),
                    "MessageBody": body,
                    "MessageGroupId": _group_for(meid),
                    "MessageDeduplicationId": _dedup_for(message_hash),
                })
                meta.append({
                    "partition_date": partition_date,
                    "meid": meid,
                    "message_hash": message_hash,
                })

                if len(batch) == BATCH_SIZE:
                    ok = flush_batch(batch, meta)
                    sent_ts = datetime.utcnow()
                    for m in ok:
                        success_rows.append({
                            "partition_date": m["partition_date"],
                            "meid": m["meid"],
                            "message_hash": m["message_hash"],
                            "run_id": RUN_ID,
                            "sent_ts": sent_ts,
                        })
                    batch, meta = [], []

            if batch:
                ok = flush_batch(batch, meta)
                sent_ts = datetime.utcnow()
                for m in ok:
                    success_rows.append({
                        "partition_date": m["partition_date"],
                        "meid": m["meid"],
                        "message_hash": m["message_hash"],
                        "run_id": RUN_ID,
                        "sent_ts": sent_ts,
                    })

            yield pd.DataFrame(success_rows)

    send_log_df = df.mapInPandas(send_map_in_pandas, schema=sendlog_schema)
    send_log_df.write.mode("append").saveAsTable(SENDLOG_TABLE)

    spark.sql(f"""
    MERGE INTO {OUTBOX_TABLE} o
    USING (
      SELECT DISTINCT partition_date, meid, message_hash
      FROM {SENDLOG_TABLE}
            WHERE partition_date = '{TARGET_PARTITION}'
        AND run_id = '{RUN_ID}'
    ) s
    ON  o.partition_date = s.partition_date
    AND o.meid           = s.meid
    AND o.message_hash   = s.message_hash
    WHEN MATCHED THEN UPDATE SET
      o.status = 'SENT',
      o.sent_ts = current_timestamp()
    """)

    sent_this_run = spark.sql(f"""
    SELECT COUNT(1) AS c
    FROM {SENDLOG_TABLE}
        WHERE partition_date = '{TARGET_PARTITION}'
      AND run_id = '{RUN_ID}'
    """).first()["c"]

    remaining_pending = spark.sql(f"""
    SELECT COUNT(1) AS c
    FROM {OUTBOX_TABLE}
        WHERE partition_date = '{TARGET_PARTITION}'
      AND status = 'PENDING'
    """).first()["c"]

    print(f"✅ DONE. Sent this run={sent_this_run:,} | Remaining pending={remaining_pending:,} | RunId={RUN_ID}")

    # ------------------------------------------------------------------
    # Slack notification to dmp-info
    # ------------------------------------------------------------------
    import urllib.request

    total_users_in_source = spark.sql(f"""
        SELECT COUNT(DISTINCT user_id) AS c
        FROM {SOURCE_TABLE}
        WHERE partition_date = '{TARGET_PARTITION}'
          AND user_id IS NOT NULL
          AND post_id IS NOT NULL
          AND size(post_id) > 0
    """).first()["c"]

    important_posts_str = ", ".join(IMPORTANT_POST_IDS[:5]) if IMPORTANT_POST_IDS else "None"

    slack_payload = json.dumps({
        "text": (
            f"*OneApp Content Recommender → SQS*\n"
            f"```"
            f"Environment : {PROFILE_CLEAN}\n"
            f"Run Date    : {TARGET_PARTITION}\n"
            f"Messages Sent      : {sent_this_run:,}\n"
            f"Remaining Pending  : {remaining_pending:,}\n"
            f"Total Users (source): {total_users_in_source:,}\n"
            f"Important Posts     : {important_posts_str}\n"
            f"Queue       : {QUEUE_URL}\n"
            f"Run ID      : {RUN_ID}"
            f"```"
        )
    })

    slack_webhook = dbutils.secrets.get(scope="dmp_secrets", key="Slack-dmp-info")
    req = urllib.request.Request(
        slack_webhook,
        data=slack_payload.encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            print(f"[INFO] Slack notification sent. Status={resp.status}")
    except Exception as e:
        print(f"[WARN] Slack notification failed: {e}")

else:
    raise ValueError(f"MODE must be 'PRINT' or 'SEND'. Got: {MODE!r}")